# Lab 05: RAG with Bedrock Knowledge Bases

## Overview
In this lab you will build an end-to-end Retrieval-Augmented Generation (RAG) pipeline using Amazon Bedrock Knowledge Bases. You will create a Knowledge Base connected to an S3 data source, ingest documents, and query with both the RetrieveAndGenerate and Retrieve APIs. You will also compare this managed RAG approach with the custom vector search pipeline from Lab 04.

## Learning Objectives
- Create a Bedrock Knowledge Base with an OpenSearch Serverless vector store
- Configure an S3 data source and run document ingestion
- Use the RetrieveAndGenerate API for fully managed RAG
- Use the Retrieve API for custom generation workflows
- Understand metadata filtering and retrieval configuration
- Compare managed RAG vs custom RAG approaches

## Exam Domain
**Building Generative AI Applications (30%)** — This lab covers Task 2.1 (RAG solutions) focusing on the managed Knowledge Bases service: creating knowledge bases, configuring data sources, ingestion pipelines, and the RetrieveAndGenerate vs Retrieve APIs.

## Architecture Diagram
![Lab 05 Architecture](../assets/diagrams/lab-05-rag-knowledge-bases.png)

In [ ]:
%pip install boto3 langchain-aws langchain --quiet

In [ ]:
import boto3, json, os, time

REGION = "us-east-1"

# Environment detection
if os.environ.get("AWS_DEFAULT_REGION") and os.path.exists("/opt/ml"):
    session = boto3.Session(region_name=REGION)
    print("Running in SageMaker Studio")
else:
    session = boto3.Session(region_name=REGION)
    print("Running locally")

bedrock_agent = session.client("bedrock-agent")
bedrock_agent_runtime = session.client("bedrock-agent-runtime")
bedrock_runtime = session.client("bedrock-runtime")
s3 = session.client("s3")
sts = session.client("sts")
aoss = session.client("opensearchserverless")

ACCOUNT_ID = sts.get_caller_identity()["Account"]
BUCKET = f"aws-genai-lab-{ACCOUNT_ID}"
BEDROCK_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/genai-lab-bedrock-role"

# Get OpenSearch collection ARN
collections = aoss.batch_get_collection(names=["genai-lab-vectors"])
COLLECTION_ARN = collections["collectionDetails"][0]["arn"]

print(f"Account:        {ACCOUNT_ID}")
print(f"S3 bucket:      {BUCKET}")
print(f"Bedrock role:   {BEDROCK_ROLE_ARN}")
print(f"Collection ARN: {COLLECTION_ARN}")

## A. Create a Knowledge Base

A **Bedrock Knowledge Base** is a managed RAG service that automates the entire pipeline: chunking documents, generating embeddings, storing them in a vector database, and retrieving relevant context at query time. You provide the configuration — Bedrock handles the rest.

The Knowledge Base requires three pieces of configuration:

1. **Knowledge Base configuration** — specifies the embedding model (Titan Embeddings V2) and the type of knowledge base (VECTOR)
2. **Storage configuration** — tells Bedrock where to store the vector embeddings (our OpenSearch Serverless collection)
3. **IAM role** — grants Bedrock permission to read from S3, invoke the embedding model, and write to the vector store

This is fundamentally different from the manual approach in Lab 04, where we wrote code for every step. Here, you declare *what* you want and the service orchestrates the pipeline.

In [ ]:
kb = bedrock_agent.create_knowledge_base(
    name="genai-lab-knowledge-base",
    roleArn=BEDROCK_ROLE_ARN,
    knowledgeBaseConfiguration={
        "type": "VECTOR",
        "vectorKnowledgeBaseConfiguration": {
            "embeddingModelArn": f"arn:aws:bedrock:{REGION}::foundation-model/amazon.titan-embed-text-v2:0"
        }
    },
    storageConfiguration={
        "type": "OPENSEARCH_SERVERLESS",
        "opensearchServerlessConfiguration": {
            "collectionArn": COLLECTION_ARN,
            "vectorIndexName": "kb-vectors",
            "fieldMapping": {
                "vectorField": "embedding",
                "textField": "text",
                "metadataField": "metadata"
            }
        }
    }
)
KB_ID = kb["knowledgeBase"]["knowledgeBaseId"]
print(f"Knowledge Base created: {KB_ID}")
print(f"Status: {kb['knowledgeBase']['status']}")

### Understanding the Configuration

| Parameter | Purpose |
|-----------|---------|
| `type: VECTOR` | Tells Bedrock this is a vector-based knowledge base (as opposed to structured/graph) |
| `embeddingModelArn` | The foundation model used to convert text chunks into vector embeddings |
| `collectionArn` | The OpenSearch Serverless collection that stores the vectors |
| `vectorIndexName` | The index within the collection where vectors are written (`kb-vectors`) |
| `vectorField` / `textField` / `metadataField` | Maps Bedrock's internal fields to the OpenSearch index field names |

The `fieldMapping` tells Bedrock which OpenSearch fields to use for storing each piece of data. Bedrock creates the index automatically if it does not already exist.

## B. Configure Data Source and Ingest Documents

A **data source** tells the Knowledge Base where to find documents. Bedrock supports S3, Confluence, SharePoint, Salesforce, and web crawlers. Here we point it at our S3 bucket's `whitepapers/` prefix, where `setup-resources.py` uploaded AWS whitepapers.

When you start an **ingestion job**, Bedrock:
1. Reads documents from the data source
2. Splits them into chunks (default: ~300 tokens with 20% overlap)
3. Generates embeddings using the configured model
4. Stores the vectors and metadata in the vector store

This entire pipeline replaces the manual chunking, embedding, and indexing code we wrote in Lab 04.

In [ ]:
# Create S3 data source pointing to the whitepapers folder
ds = bedrock_agent.create_data_source(
    knowledgeBaseId=KB_ID,
    name="aws-whitepapers",
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": f"arn:aws:s3:::{BUCKET}",
            "inclusionPrefixes": ["whitepapers/"]
        }
    }
)
DS_ID = ds["dataSource"]["dataSourceId"]
print(f"Data source created: {DS_ID}")
print(f"Status: {ds['dataSource']['status']}")

### Start Ingestion

Ingestion is asynchronous. We start the job and then poll until it completes. Depending on the number and size of documents, this typically takes 1-5 minutes.

In [ ]:
# Start ingestion job
job = bedrock_agent.start_ingestion_job(knowledgeBaseId=KB_ID, dataSourceId=DS_ID)
JOB_ID = job["ingestionJob"]["ingestionJobId"]
print(f"Ingestion job started: {JOB_ID}")

# Poll until complete
while True:
    status = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=KB_ID,
        dataSourceId=DS_ID,
        ingestionJobId=JOB_ID
    )
    job_status = status["ingestionJob"]["status"]
    print(f"  Status: {job_status}")

    if job_status in ("COMPLETE", "FAILED", "STOPPED"):
        break
    time.sleep(15)

# Show ingestion statistics
stats = status["ingestionJob"].get("statistics", {})
print(f"\nIngestion complete!")
print(f"  Documents scanned:  {stats.get('numberOfDocumentsScanned', 'N/A')}")
print(f"  Documents indexed:  {stats.get('numberOfNewDocumentsIndexed', 0) + stats.get('numberOfModifiedDocumentsIndexed', 0)}")
print(f"  Documents failed:   {stats.get('numberOfDocumentsFailed', 'N/A')}")

## C. RetrieveAndGenerate — Managed RAG

The **RetrieveAndGenerate** API is Bedrock's fully managed RAG endpoint. In a single API call it:
1. Embeds your query using the Knowledge Base's embedding model
2. Performs vector search against the configured vector store
3. Constructs a prompt with retrieved context
4. Sends that prompt to the specified foundation model
5. Returns the generated answer with source citations

This is the simplest way to add RAG to an application — you do not need to manage prompts, context windows, or retrieval logic.

In [ ]:
# Query 1: Well-Architected Framework pillars
response = bedrock_agent_runtime.retrieve_and_generate(
    input={"text": "What are the six pillars of the AWS Well-Architected Framework?"},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": KB_ID,
            "modelArn": f"arn:aws:bedrock:{REGION}::foundation-model/us.anthropic.claude-sonnet-4-5-20250929-v1:0"
        }
    }
)

print("Answer:")
print(response["output"]["text"])
print("\nSources:")
for citation in response.get("citations", []):
    for ref in citation.get("retrievedReferences", []):
        loc = ref.get("location", {}).get("s3Location", {})
        print(f"  - {loc.get('uri', 'unknown')}")

In [ ]:
# Query 2: Amazon Bedrock capabilities
response = bedrock_agent_runtime.retrieve_and_generate(
    input={"text": "What is Amazon Bedrock and what are its key capabilities?"},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": KB_ID,
            "modelArn": f"arn:aws:bedrock:{REGION}::foundation-model/us.anthropic.claude-sonnet-4-5-20250929-v1:0"
        }
    }
)

print("Answer:")
print(response["output"]["text"])
print("\nSources:")
for citation in response.get("citations", []):
    for ref in citation.get("retrievedReferences", []):
        loc = ref.get("location", {}).get("s3Location", {})
        print(f"  - {loc.get('uri', 'unknown')}")

In [ ]:
# Query 3: Security best practices
response = bedrock_agent_runtime.retrieve_and_generate(
    input={"text": "What are the best practices for securing generative AI applications on AWS?"},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": KB_ID,
            "modelArn": f"arn:aws:bedrock:{REGION}::foundation-model/us.anthropic.claude-sonnet-4-5-20250929-v1:0"
        }
    }
)

print("Answer:")
print(response["output"]["text"])
print("\nSources:")
for citation in response.get("citations", []):
    for ref in citation.get("retrievedReferences", []):
        loc = ref.get("location", {}).get("s3Location", {})
        print(f"  - {loc.get('uri', 'unknown')}")

## D. Retrieve Only — Custom Generation

The **Retrieve** API returns raw document chunks without generating a response. This gives you full control over the generation step — you can build custom prompts, use a different model for generation, chain multiple retrieval calls, or perform post-processing on the retrieved context before passing it to a model.

**When to use Retrieve vs RetrieveAndGenerate:**

| Use Case | API |
|----------|-----|
| Simple Q&A with citations | RetrieveAndGenerate |
| Custom prompt templates | Retrieve |
| Multi-step reasoning pipelines | Retrieve |
| Different retrieval and generation models | Retrieve |
| Pre-processing or filtering retrieved chunks | Retrieve |
| Maximum simplicity, minimum code | RetrieveAndGenerate |

### Step 1: Retrieve Relevant Chunks

We call the Retrieve API to get the top 5 document chunks most relevant to our query. Each result includes the chunk text, a relevance score, and metadata about the source document.

In [ ]:
# Retrieve chunks without generation
results = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={"text": "What is Amazon Bedrock?"},
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "numberOfResults": 5
        }
    }
)

# Inspect the retrieved chunks
for i, result in enumerate(results["retrievalResults"]):
    score = result.get("score", 0)
    text = result["content"]["text"][:200]
    source = result.get("location", {}).get("s3Location", {}).get("uri", "unknown")
    print(f"Chunk {i+1} (score: {score:.4f})")
    print(f"  Source: {source}")
    print(f"  Text:   {text}...")
    print()

### Step 2: Build a Custom Prompt and Generate

Now we assemble the retrieved chunks into a custom prompt. This gives us full control over the prompt template — we can add specific instructions, format requirements, or persona guidance that RetrieveAndGenerate does not support.

In [ ]:
# Build custom prompt with retrieved context
context = "\n\n".join([r["content"]["text"] for r in results["retrievalResults"]])

prompt = f"""Based on the following context from AWS documentation, answer the question.
If the context does not contain enough information, say so clearly.

Context:
{context}

Question: What is Amazon Bedrock and what are its key features?

Provide a concise answer in 3-4 sentences."""

response = bedrock_runtime.converse(
    modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    messages=[{"role": "user", "content": [{"text": prompt}]}],
    inferenceConfig={"maxTokens": 512}
)

print("Custom RAG Answer:")
print(response["output"]["message"]["content"][0]["text"])

## E. Appendix — LangChain Integration (Optional)

> **This section is optional.** It demonstrates how LangChain wraps the same Bedrock Knowledge Base APIs into its retriever abstraction. This is useful if you are building applications with LangChain's chain/agent framework, but the native Boto3 APIs shown above are sufficient for most use cases and for the exam.

LangChain's `AmazonKnowledgeBasesRetriever` provides a standard retriever interface that integrates with LangChain chains. Under the hood it calls the same Retrieve API we used in Section D.

In [ ]:
# Optional: LangChain retriever wrapping Bedrock Knowledge Bases
from langchain_aws import AmazonKnowledgeBasesRetriever

retriever = AmazonKnowledgeBasesRetriever(
    knowledge_base_id=KB_ID,
    retrieval_config={"vectorSearchConfiguration": {"numberOfResults": 5}},
)

docs = retriever.invoke("What is the Well-Architected Framework?")

print(f"Retrieved {len(docs)} documents via LangChain:\n")
for i, doc in enumerate(docs):
    score = doc.metadata.get("score", "N/A")
    score_str = f"{score:.4f}" if isinstance(score, float) else str(score)
    print(f"Document {i+1} (score: {score_str})")
    print(f"  {doc.page_content[:200]}...")
    print()

In [ ]:
# Optional: LangChain RAG chain with Knowledge Base retriever
from langchain_aws import ChatBedrock
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatBedrock(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name=REGION
)

template = """Answer the question based only on the following context.
If the context does not contain enough information, say so.

Context:
{context}

Question: {question}"""

prompt = ChatPromptTemplate.from_template(template)

# Build the chain: retrieve -> format -> generate
chain = (
    {"context": retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)),
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = chain.invoke("How does Amazon Bedrock support responsible AI?")
print("LangChain RAG Answer:")
print(answer)

## Cleanup

Delete the Knowledge Base and its data source. **Do not** delete the OpenSearch Serverless collection itself — it is shared infrastructure used by other labs and managed by `setup-resources.py`.

In [ ]:
# Delete data source first, then the Knowledge Base
bedrock_agent.delete_data_source(knowledgeBaseId=KB_ID, dataSourceId=DS_ID)
print(f"Deleted data source: {DS_ID}")

bedrock_agent.delete_knowledge_base(knowledgeBaseId=KB_ID)
print(f"Deleted Knowledge Base: {KB_ID}")

print("\nKnowledge Base and data source deleted.")
print("OpenSearch Serverless collection 'genai-lab-vectors' is preserved for other labs.")

## Key Takeaways

1. **Bedrock Knowledge Bases automate the RAG pipeline** — document ingestion, chunking, embedding, and vector storage are fully managed, replacing dozens of lines of custom code with a single configuration
2. **RetrieveAndGenerate is the fastest path to RAG** — one API call handles retrieval, prompt construction, and generation with automatic source citations
3. **Retrieve API enables custom workflows** — when you need custom prompts, multi-step reasoning, or different generation models, Retrieve gives you raw chunks to build your own pipeline
4. **Ingestion is incremental** — re-running ingestion only processes new or modified documents, making it efficient for production data sources that change over time
5. **Managed RAG trades flexibility for simplicity** — you cannot customize the retrieval prompt template in RetrieveAndGenerate, but you avoid managing embeddings, vector indices, and prompt engineering

## Key Concepts

| Concept | Definition |
|---------|-----------|
| Knowledge Base | A managed Bedrock resource that connects a data source to a vector store and an embedding model, automating the entire RAG ingestion and retrieval pipeline |
| Data Source | A connector that tells the Knowledge Base where to find documents (S3, Confluence, SharePoint, web crawler); each Knowledge Base can have multiple data sources |
| Ingestion Job | An asynchronous process that reads documents from a data source, chunks them, generates embeddings, and stores the vectors in the configured vector store |
| RetrieveAndGenerate | A single API call that performs retrieval from the Knowledge Base and generation with a foundation model, returning an answer with source citations |
| Retrieve | An API that returns raw document chunks from the Knowledge Base without generation, enabling custom prompt construction and multi-step workflows |
| Vector Store | The backend database (OpenSearch Serverless, Pinecone, or Redis) that stores and searches the embedding vectors created during ingestion |
| Chunking Strategy | The method used to split documents into smaller segments before embedding; Bedrock supports fixed-size, default, semantic, and hierarchical chunking strategies |

## Exam Preparation — Common Exam Question Patterns

**Q: What is the difference between RetrieveAndGenerate and Retrieve APIs in Bedrock Knowledge Bases?**
A: RetrieveAndGenerate performs end-to-end RAG in a single call — it retrieves relevant chunks, constructs a prompt, sends it to a foundation model, and returns the generated answer with citations. Retrieve only returns the relevant document chunks, giving developers full control over prompt construction, model selection, and post-processing. Use RetrieveAndGenerate for simple Q&A and Retrieve when you need custom prompts or multi-step workflows.

**Q: What vector stores does Amazon Bedrock Knowledge Bases support?**
A: Bedrock Knowledge Bases supports Amazon OpenSearch Serverless, Amazon Aurora PostgreSQL (with pgvector), Pinecone, and Redis Enterprise Cloud as vector stores. OpenSearch Serverless is the most commonly used option because it is fully managed and scales automatically within AWS.

**Q: How does Bedrock Knowledge Bases handle document updates?**
A: When you re-run an ingestion job, Bedrock performs incremental ingestion — it identifies new and modified documents in the data source, processes only those documents, and updates the vector store accordingly. Deleted documents are also removed from the index. This makes it efficient for production data sources that change frequently.

**Q: What chunking strategies are available in Bedrock Knowledge Bases?**
A: Bedrock supports four chunking strategies: default (automatic ~300 tokens), fixed-size (configurable token count with overlap), semantic (groups related sentences), and hierarchical (parent-child chunks for better context). The choice depends on your document structure — semantic chunking works well for narrative documents, while fixed-size is predictable and efficient for uniform content.

**Q: When should you use Bedrock Knowledge Bases vs building a custom RAG pipeline?**
A: Use Knowledge Bases when you want a fully managed RAG solution with minimal code, automatic document syncing, and built-in chunking and embedding. Build a custom pipeline when you need custom chunking logic, specialized embedding models not available in Bedrock, complex pre-processing steps, or fine-grained control over the retrieval and generation stages. Knowledge Bases reduce operational overhead but offer less flexibility than custom implementations.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| Titan Embeddings V2 — Ingestion (document chunking + embedding) | ~50-100 chunks, ~25K tokens | < $0.01 |
| Claude Sonnet 4.5 — Section C (RetrieveAndGenerate, 3 queries) | ~3 API calls, ~6K tokens | ~$0.10 |
| Claude Sonnet 4.5 — Section D (custom generation) | ~1 API call, ~2K tokens | ~$0.03 |
| Claude Sonnet 4.5 — Section E (LangChain chain, optional) | ~1 API call, ~2K tokens | ~$0.03 |
| OpenSearch Serverless — OCU time | ~2 OCUs x 1.25 hours | ~$2-3 |
| **Total** | | **~$2-3** |

The dominant cost is OpenSearch Serverless compute (OCU) time, billed per OCU-hour while the collection is active. The collection was created by `setup-resources.py` and will continue accruing charges until deleted via `cleanup-all.py`. Bedrock API costs for this lab are minimal.